In [5]:
"""FINAL Section 6 - unit-level Monte Carlo investment model (uses findings from Sections 1-4; Section 4: plant rework spend is recorded on ALL units, so it is NOT counted as avoidable savings in the base case).
Place next to Competition_Dataset_Final_1.xlsx.  python section6_final.py"""
%matplotlib inline
import numpy as np, pandas as pd, itertools, copy
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
df = pd.read_excel("DataSets/Competition_Dataset_Final.xlsx"); n=len(df); rng=np.random.default_rng(7)
fd=df["Failure Description"].values
leak=np.isin(fd,["Refrigerant Leak","Service Valve Leak","Suction Leak"]); ctrl=np.isin(fd,["Communication Fault","Sensor Fault"])
loose=fd=="Loose Power Wiring"; fan=fd=="Condenser Fan Backwards"; legacy=(df["Product Family"]=="LegacyX").values
supB=(df["Coil Supplier"]=="Supplier B").values; allu=np.ones(n,bool)
RC=df["Repair Cost Field Service"].values.astype(float); WC=df["Warranty Cost"].values.astype(float); MC=df["Material Cost Rework"].values.astype(float)
SCEN={"Repair only (warranty overlaps)":RC,"Repair + warranty (additive)":RC+WC}
DET={4,7,8}   # detection options: need a detected defect to be FIXED (Section 2: defects already detected; Section 4: rework shows no benefit)
OPT={1:("Braze training",50e3,2,"L",.10,leak),2:("Smart torque tools",100e3,3,"L",.30,loose),
 3:("Wiring poka-yoke",125e3,4,"L",.40,np.isin(fd,["Sensor Fault","Communication Fault"])|fan),
 4:("Predictive dashboard",150e3,6,"M",.15,allu),5:("Design validation",200e3,6,"M",.20,ctrl|legacy),
 6:("Vision inspection",275e3,6,"M",.35,loose|fan),7:("Auto leak detection",350e3,8,"L",.40,leak),
 8:("Extra EOLT station",450e3,8,"M",.15,allu),9:("Supplier quality",175e3,4,"M",.25,leak&supB),10:("LegacyX refresh",500e3,12,"H",.35,legacy)}
LO={"L":.6,"M":.4,"H":.2}
def P0(): return dict(m={k:v[2] for k,v in OPT.items()},r={k:v[4] for k,v in OPT.items()},mask={k:v[5] for k,v in OPT.items()},cost={k:v[1] for k,v in OPT.items()},plant=False)
def emean(ev=True): return {k:(LO[v[3]]+1)/2*(0.5 if (ev and k in DET) else 1) for k,v in OPT.items()}
def draw_e(ev=True): return {k:rng.uniform(LO[v[3]],1)*(rng.uniform(0,1) if (ev and k in DET) else 1) for k,v in OPT.items()}
def draw_w(): return rng.multinomial(n,np.full(n,1/n)).astype(float)
def rate(live,C,e,w,P):
    if not live: return 0.0
    fs=np.ones(n); ps=np.ones(n)
    for k in live:
        p=P["mask"][k]*P["r"][k]*e[k]; fs=fs*(1-p)
        if k not in DET: ps=ps*(1-p)           # plant rework material only avoided if the defect is PREVENTED
    out=((1-fs)*C*w).sum()
    if P["plant"]: out+=((1-ps)*MC*w).sum()
    return out
def cum(S,T,C,e,w,P):
    bps=sorted(set([0,T]+[min(T,P["m"][k]) for k in S])); t=0.0
    for a,b in zip(bps[:-1],bps[1:]): t+=rate([k for k in S if P["m"][k]<=a],C,e,w,P)*(b-a)/12
    return t
def cost_of(S,cm=1.0): return cm*sum(OPT[k][1] for k in S)
def net(S,T,C,e,w,P,cm=1.0): return cum(S,T,C,e,w,P)-cost_of(S,cm)
def path_fn(S,C,e,w,P):
    bps=sorted(set([0]+[P["m"][k] for k in S])); rs=[rate([k for k in S if P["m"][k]<=a],C,e,w,P)/12 for a in bps]
    return lambda t: sum(r*(min(max(t-a,0),(bps[i+1]-a) if i+1<len(bps) else 1e9)) for i,(a,r) in enumerate(zip(bps,rs)))
W1=np.ones(n); N=300; T=np.arange(0,61); P=P0()
subs=[S for r_ in range(1,9) for S in itertools.combinations(OPT,r_) if cost_of(S)<=750e3]
if __name__=="__main__":
    EM=emean(True); sn1,sn2=list(SCEN)
    print("== PROBLEM COSTS (annual, from data)")
    for nm,mk in [("Leaks",leak),("Loose power wiring",loose),("Controls faults (Rev A)",ctrl),("Backwards fans (night)",fan)]:
        print(f"{nm:26s} units {mk.sum():4d} field repair ${RC[mk].sum()/1e3:6.0f}K warranty ${WC[mk].sum()/1e3:6.0f}K plant material ${MC[mk].sum()/1e3:5.0f}K")
    print("== SINGLE OPTIONS (evidence-adjusted, expected): ROI 1/3/5y, repair-only | additive")
    for k,(nm,c,m,rk,r,mk) in OPT.items():
        s=f"{k:2d} {nm:22s} ${c/1e3:4.0f}K {'DET' if k in DET else '   '}"
        for C in SCEN.values(): s+=" | "+"/".join(f"{100*(cum([k],12*y,C,EM,W1,P)-c)/c:5.0f}%" for y in (1,3,5))
        print(s)
    res={}
    for sn,C in SCEN.items():
        for ev in (True,False):
            em=emean(ev); rr=sorted(((net(S,36,C,em,W1,P),S) for S in subs),reverse=True)[:3]
            print(f"best 3y [{sn}] {'evidence-adj' if ev else 'stated claims':13s}:",[(S,round(v/1e3)) for v,S in rr])
    REC=None
    # choose REC = best expected 3y net under repair-only, evidence-adjusted
    C1=SCEN[sn1]
    # SELECTION RULE: minimax regret over 6 scenarios (2 warranty readings x detection options deliver 0 / evidence-adjusted / stated claims)
    em_s=emean(False); em_z={k:(0 if k in DET else v) for k,v in em_s.items()}
    scen=[(C,em) for C in SCEN.values() for em in (em_z,EM,em_s)]
    tab={S:[net(S,36,C,em,W1,P) for C,em in scen] for S in subs}; bst=[max(tab[S][i] for S in subs) for i in range(6)]
    reg={S:max((bst[i]-tab[S][i])/bst[i] for i in range(6)) for S in subs}
    REC=min(subs,key=lambda S:reg[S]); print("\nREC (min max-regret)",REC,cost_of(REC),"max regret %.0f%%"%(100*reg[REC]))
    print("highest expected 3y net alone would pick",max(subs,key=lambda S:net(S,36,C1,EM,W1,P)),"(max regret %.0f%%)"%(100*reg[max(subs,key=lambda S:net(S,36,C1,EM,W1,P))]))
    out={}
    for sn,C in SCEN.items():
        top=[S for _,S in sorted(((net(S,36,C,EM,W1,P),S) for S in subs),reverse=True)[:12]]
        if REC not in top: top.append(REC)
        M=np.zeros((N,len(top))); paths=np.zeros((N,61)); opt3=np.zeros((N,10))
        for i in range(N):
            e,w=draw_e(True),draw_w()
            for j,S in enumerate(top): M[i,j]=net(S,36,C,e,w,P)
            f=path_fn(REC,C,e,w,P); paths[i]=[f(t)-cost_of(REC) for t in T]
            for k in OPT: opt3[i,k-1]=cum([k],36,C,e,w,P)/OPT[k][1]-1
        best=M.argmax(1); out[sn]=dict(top=top,M=M,paths=paths,opt3=opt3)
        print("\n==",sn)
        for j,S in sorted(enumerate(top),key=lambda x:-M[:,x[0]].mean())[:6]:
            print(S,f"cost {cost_of(S)/1e3:.0f}K mean3y {M[:,j].mean()/1e3:.0f}K p5 {np.percentile(M[:,j],5)/1e3:.0f}K p95 {np.percentile(M[:,j],95)/1e3:.0f}K P(net>0) {(M[:,j]>0).mean():.2f} P(best) {(best==j).mean():.2f}")
        pa=paths
        for y in (1,3,5):
            v=pa[:,12*y]; print(f"REC yr{y}: net {v.mean()/1e3:.0f}K  p5 {np.percentile(v,5)/1e3:.0f}K p95 {np.percentile(v,95)/1e3:.0f}K ROI {100*v.mean()/cost_of(REC):.0f}% P>0 {(v>0).mean():.2f}")
        pm=pa.mean(0); print("payback month:",int(np.argmax(pm>0)) if (pm>0).any() else None)
    # tornado
    def tv(Pp=P,C=C1,e=EM,cm=1.0): return net(REC,36,C,e,W1,Pp,cm)/1e3
    base=tv(); items=[("Warranty counted as additive",base,tv(C=SCEN[sn2])),
     ("Detection options deliver nothing",tv(e={k:(0 if k in DET else v) for k,v in EM.items()}),base),
     ("Detection options deliver stated claims",base,tv(e=emean(False))),
     ("Delivered effectiveness (50% / 100% of claim)",tv(e={k:v*.5 for k,v in EM.items()}),tv(e={k:(1.0 if k not in DET else .5) for k in EM})),
     ("Plant rework savings counted (upside)",base,tv(Pp=dict(P,plant=True))),("Investment cost +25%",tv(cm=1.25),base)]
    Pd=copy.deepcopy(P); Pd["m"]={k:v+3 for k,v in P["m"].items()}; items.append(("Implementation 3 months late",tv(Pp=Pd),base))
    print("\nTORNADO base %.0fK"%base); [print(f"{a:48s} {l:8.0f} {h:8.0f}") for a,l,h in items]
    # ---- figures
    plt.rcParams.update({"font.size":15,"axes.spines.top":False,"axes.spines.right":False,"figure.dpi":150}); NAVY,TEAL,GREY,ORG="#1F3A5F","#2A9D8F","#B8BEC6","#E07A1F"
    # fig0 problem costs
    pr=[("Leaks",leak),("Loose power wiring",loose),("Controls faults\n(Rev A)",ctrl),("Backwards fans\n(night shift)",fan)]
    fig,ax=plt.subplots(figsize=(13,6.4)); x=np.arange(4); wd=.27
    ax.bar(x-wd,[RC[m].sum()/1e6 for _,m in pr],wd,color=NAVY,label="Field repair"); ax.bar(x,[WC[m].sum()/1e6 for _,m in pr],wd,color=TEAL,label="Warranty (counted only if additive)"); ax.bar(x+wd,[MC[m].sum()/1e6 for _,m in pr],wd,color=ORG,label="Plant rework material (recorded on all units; not counted as savings)")
    for i,(_,m) in enumerate(pr): ax.text(i-wd,RC[m].sum()/1e6+.05,f"${RC[m].sum()/1e6:.2f}M",ha="center",fontsize=12)
    ax.set_xticks(x); ax.set_xticklabels([a for a,_ in pr]); ax.set_ylabel("Annual cost ($M)"); ax.legend(frameon=False); plt.tight_layout(); plt.show()
    # fig1
    fig,ax=plt.subplots(figsize=(13,7.2)); y=np.arange(10); order=sorted(OPT,key=lambda k:out[sn2]["opt3"][:,k-1].mean())
    for off,(sn,col) in zip((-.2,.2),zip(SCEN,(TEAL,NAVY))):
        m=[out[sn]["opt3"][:,k-1] for k in order]
        ax.barh(y+off,[v.mean() for v in m],.38,color=col,label=sn,xerr=[[v.mean()-np.percentile(v,5) for v in m],[np.percentile(v,95)-v.mean() for v in m]],error_kw=dict(ecolor="#555",lw=1))
    ax.set_yticks(y); ax.set_yticklabels([("★ " if k in REC else "")+OPT[k][0]+(" †" if k in DET else "")+f"  (${OPT[k][1]/1e3:.0f}K)" for k in order])
    ax.set_xlabel("3-year net return per $1 invested (bars: mean; whiskers: 5th–95th pct.)"); ax.axvline(0,color="k",lw=1); ax.legend(frameon=False,loc="lower right"); ax.set_title("★ recommended    † detection option, benefit discounted for evidence (Sections 2 & 4)",loc="left",fontsize=12)
    plt.tight_layout(); plt.show()
    # fig2
    fig,axs=plt.subplots(1,2,figsize=(14,6.2))
    for ax,sn,col in zip(axs,SCEN,(TEAL,NAVY)):
        pa=out[sn]["paths"]/1e6; mu=pa.mean(0); ax.fill_between(T/12,np.percentile(pa,5,0),np.percentile(pa,95,0),color=col,alpha=.2,label="5th–95th pct."); ax.plot(T/12,mu,color=col,lw=3,label="Expected"); ax.axhline(0,color="k",lw=1)
        for yr in (1,3,5): ax.scatter(yr,mu[12*yr],color=ORG,s=90,zorder=5); ax.annotate(f"${mu[12*yr]:.1f}M",(yr,mu[12*yr]),textcoords="offset points",xytext=(-8,12),ha="right",fontsize=14,color=ORG,fontweight="bold")
        ax.set_title(sn,fontsize=15); ax.set_xlabel("Years after investment"); ax.set_xticks(range(6)); ax.set_ylabel("Cumulative net benefit ($M)")
    axs[0].legend(frameon=False,loc="upper left"); plt.tight_layout(); plt.show()
    # fig3
    fig,axs=plt.subplots(1,2,figsize=(14,6.2))
    for ax,sn,col in zip(axs,SCEN,(TEAL,NAVY)):
        o=out[sn]; M=o["M"]; idx=sorted(range(len(o["top"])),key=lambda j:-M[:,j].mean())[:6][::-1]
        ax.barh(range(6),[M[:,j].mean()/1e6 for j in idx],color=[ORG if o["top"][j]==REC else col for j in idx],xerr=[[(M[:,j].mean()-np.percentile(M[:,j],5))/1e6 for j in idx],[(np.percentile(M[:,j],95)-M[:,j].mean())/1e6 for j in idx]],error_kw=dict(ecolor="#555"))
        ax.set_yticks(range(6)); ax.set_yticklabels(["+".join(map(str,o["top"][j]))+f"  (${cost_of(o['top'][j])/1e3:.0f}K)" for j in idx]); ax.set_title(sn,fontsize=15); ax.set_xlabel("3-year net benefit ($M)")
    plt.tight_layout(); plt.show()
    # fig4
    it=sorted(items,key=lambda x:abs(x[2]-x[1])); fig,ax=plt.subplots(figsize=(13,6.6))
    for i,(a,l,h) in enumerate(it):
        ax.barh(i,l-base,left=base,color=ORG); ax.barh(i,h-base,left=base,color=TEAL)
    ax.axvline(base,color="k",lw=1.5); ax.set_yticks(range(len(it))); ax.set_yticklabels([a for a,_,_ in it]); ax.set_xlabel("3-year net benefit of recommended portfolio ($K)"); ax.set_title(f"Base case: ${base/1e3:.1f}M (repair-only, evidence-adjusted)",loc="left",fontsize=13)
    plt.tight_layout(); plt.show()


== PROBLEM COSTS (annual, from data)
Leaks                      units  788 field repair $  2892K warranty $  5784K plant material $  213K
Loose power wiring         units 1744 field repair $   473K warranty $   948K plant material $  460K
Controls faults (Rev A)    units  283 field repair $   392K warranty $   785K plant material $   72K
Backwards fans (night)     units  185 field repair $    68K warranty $   136K plant material $   50K
== SINGLE OPTIONS (evidence-adjusted, expected): ROI 1/3/5y, repair-only | additive
 1 Braze training         $  50K     |   286%/ 1211%/ 2136% |  1057%/ 3833%/ 6610%
 2 Smart torque tools     $ 100K     |   -15%/  212%/  440% |   156%/  838%/ 1520%
 3 Wiring poka-yoke       $ 125K     |   -21%/  214%/  450% |   136%/  844%/ 1551%
 4 Predictive dashboard   $ 150K DET |   -33%/  235%/  503% |   101%/  904%/ 1708%
 5 Design validation      $ 200K     |   -56%/  119%/  295% |    32%/  558%/ 1085%
 6 Vision inspection      $ 275K     |   -76%/   21%/  117% 

C:\Users\ecast\AppData\Local\Temp\ipykernel_12380\542314500.py:103: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xticks(x); ax.set_xticklabels([a for a,_ in pr]); ax.set_ylabel("Annual cost ($M)"); ax.legend(frameon=False); plt.tight_layout(); plt.show()
C:\Users\ecast\AppData\Local\Temp\ipykernel_12380\542314500.py:111: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
C:\Users\ecast\AppData\Local\Temp\ipykernel_12380\542314500.py:118: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axs[0].legend(frameon=False,loc="upper left"); plt.tight_layout(); plt.show()
C:\Users\ecast\AppData\Local\Temp\ipykernel_12380\542314500.py:125: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
C:\Users\ecast\AppData\Local\Temp\ipykernel_12380\542314500.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be